Running importance analysis with Python API (Regression)
=========================================================

This notebook demonstrates running *VariantSpark* importance analysis using a **regression** random forest (`RandomForestRegressor`), as opposed to the classification variant.

`chr22-values.csv` contains a continuous response variable constructed as a weighted linear combination of two genomic variants:

$$\text{response} = 0.4 \times \texttt{22\_16051347\_G\_C} + (-0.6) \times \texttt{22\_16050984\_A\_G}$$

We would therefore expect these two positions to rank as the most important variables in the analysis.


Step 1: Create a spark session with VariantSpark jar attached.

In [1]:
import varspark as vs
from pyspark.sql import SparkSession 
spark = vs.configure_spark(
    SparkSession.builder.config('spark.jars', vs.find_jar())
).getOrCreate()

26/03/27 13:08:02 WARN Utils: Your hostname, RADON-BH resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/27 13:08:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/27 13:08:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Step 2: Create a `VarsparkContext` using `SparkSession` object (here injected as `spark`):

In [2]:
vc = vs.VarsparkContext(spark, silent = True)

Step 3: Load the features `fs` and continuous response `ls`.  and the `chr22-values.csv` file.

In [3]:
fs = vc.import_vcf('../../data/chr22_1000.vcf')
ls = vc.load_response('../../data/chr22-values.csv', 'response')

../../data/chr22_1000.vcf is loading to spark RDD, isBGZFile: false


Step 4: Fit a regression RF model using `RandomForestRegressor`.

In [4]:
rf = vs.RandomForestRegressor(vc, mtry_fraction=0.2, min_node_size=5, max_depth=10, seed=13)
rf.fit_trees(fs, ls, n_trees=500, batch_size=20)

Step 5: Retrieve top important variables and the full importance table as a pandas DataFrame:

In [5]:
ia = rf.importance_analysis()
top_variables = ia.important_variables(limit=10, normalized=True)

In [6]:
# return pandas dataframe of variable importances
importance = ia.variable_importance(normalized = True)
# sort by importance
importance = importance.sort_values('importance', ascending=False)
importance.head(10)

,variant_id,importance,splitCount
253,22_16051347_G_C,0.335648,1029
969,22_16051497_A_G,0.335273,969
975,22_16053791_C_A,0.102179,377
1229,22_16052239_A_G,0.046474,258
1343,22_16052513_G_C,0.030836,158
1345,22_16054667_C_G,0.016347,101
1113,22_16051453_A_C,0.016173,175
384,22_16051249_T_C,0.015870,161
1,22_16052618_G_A,0.015817,94
1797,22_16053659_A_C,0.014533,187


Step 6: Print the top variables by importance.

In [7]:
print("%s\t%s" % ('Variable', 'Importance'))
for var_and_imp in top_variables.values:
    print("%s\t%s" % tuple(var_and_imp))

Variable	Importance
22_16051347_G_C	0.3356480444760767
22_16051497_A_G	0.3352728720768356
22_16053791_C_A	0.10217926072037244
22_16052239_A_G	0.04647431346328525
22_16052513_G_C	0.030835843723505327
22_16054667_C_G	0.016346615429239177
22_16051453_A_C	0.016172734201582693
22_16051249_T_C	0.015869552777405163
22_16052618_G_A	0.01581671534726592
22_16053659_A_C	0.014532742233297036


For more information on using *VariantSpark* and the Python API please visit the [documentation](http://variantspark.readthedocs.io/en/latest/).